In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_order_items")
display(df.head(10))


dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code
2024-12-24,2024-12-24T11:22:14.000Z,CUST000000175243,397799,1,2000000108544,1,SGD,30,12%,2,app,PRIME5
2024-12-24,2024-12-24T11:22:14.000Z,CUST000000175243,397799,2,2000000459974,Two,SGD,256,0%,62,app,PRIME5
2024-12-24,2024-12-24T21:35:44.000Z,CUST000000227431,397800,1,2000000077475,Two,CAD,48,15%,10,web,NEW10
2024-12-24,2024-12-24T18:36:33.000Z,CUST000000052560,397801,1,2000000348407,1,GBP,38,2%,7,web,null
2024-12-24,2024-12-24T18:36:33.000Z,CUST000000052560,397801,2,2000000254968,1,GBP,26,0%,4,web,null
2024-12-24,2024-12-24T22:04:31.000Z,CUST000000168011,397802,1,2000000004679,1,INR,53906,12%,5720,app,null
2024-12-24,2024-12-24T21:53:52.000Z,CUST000000128913,397803,1,2000000313153,1,GBP,528,16%,54,web,null
2024-12-24,2024-12-24T21:53:52.000Z,CUST000000128913,397803,2,2000000352169,1,GBP,656,19%,64,web,null
2024-12-24,2024-12-24T21:53:52.000Z,CUST000000128913,397803,3,2000000292649,1,GBP,71,9%,12,web,null
2024-12-24,2024-12-24T21:53:52.000Z,CUST000000128913,397803,4,2000000441542,Two,GBP,164,8%,55,web,null


In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])

# Transformation: Convert 'Two' → 2 and cast to Integer
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

# Transformation : channel processing 
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#Transformation : Add processed time 
df = df.withColumn(
    "processed_time", F.current_timestamp()
)


In [0]:
display(df.limit(20))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,processed_time
2024-01-01,2024-01-01T21:01:14.000Z,CUST000000189601,10,1,2000000126029,1,INR,2882.0,14.0,299,Website,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T10:32:10.000Z,CUST000000119485,11,1,2000000052304,1,INR,28774.0,1.0,5147,Website,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T23:15:48.000Z,CUST000000156472,26,1,2000000332710,1,INR,2090.0,8.0,97,Mobile,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T04:21:53.000Z,CUST000000211695,51,2,2000000161365,1,INR,731.0,0.0,88,Mobile,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T23:56:03.000Z,CUST000000219795,52,1,2000000382524,1,INR,1622.0,24.0,222,Website,prime5,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T09:29:39.000Z,CUST000000165869,53,1,2000000133874,2,GBP,49.0,26.0,9,Mobile,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T10:56:03.000Z,CUST000000065340,80,2,2000000202709,1,USD,804.0,16.0,82,Mobile,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T16:05:05.000Z,CUST000000243788,84,1,2000000289786,1,INR,3450.0,0.0,414,Website,fest20,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T16:37:09.000Z,CUST000000063627,92,2,2000000304519,2,INR,2524.0,28.0,438,Website,null,2026-10-24T20:03:56.093Z
2024-01-01,2024-01-01T20:57:36.000Z,CUST000000010241,100,4,2000000230535,3,INR,6298.0,0.0,945,Mobile,null,2026-10-24T20:03:56.093Z


In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.silv_order_items")

In [0]:
df= spark.read.table("ecommerce.bronze.brz_order_returns")
display(df.limit(5))

return_ts,order_dt,order_id,reason
2024-01-08T00:52:46.000Z,2024-01-01,386,Damaged
2024-01-08T15:24:08.000Z,2024-01-01,771,Late Delivery
2024-01-08T15:52:54.000Z,2024-01-01,27,Damaged
2024-01-08T20:14:41.000Z,2024-01-02,1682,Late Delivery
2024-01-08T22:42:42.000Z,2024-01-01,1021,Wrong Item


In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.silv_order_returns")


In [0]:
df= spark.read.table("ecommerce.bronze.brz_order_shipments")
display(df.limit(5))

order_dt,shipment_id,order_id,carrier
2024-01-01,S1,1,FedEx
2024-01-01,S2,2,EcomExpress
2024-01-01,S3,3,Delhivery
2024-01-01,S4,4,EcomExpress
2024-01-01,S5,5,DHL


In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.silv_order_shipments")